# Replay-safe webhook analytics

Webhook senders retry events. Duplicate deliveries, conflicting IDs and inconsistent money handling can corrupt dashboard totals even when the endpoint appears to work.

Authenticate the raw body with HMAC and a timestamp window, validate the schema, store a canonical hash and event ID, and commit receipt plus outbox together. A separate worker atomically updates daily currency totals and marks work complete.

Rebuild source artifacts with `python scripts/run_all.py` from the repository root. This walkthrough inspects the measured run.

In [1]:
from pathlib import Path
import json
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
filename = "audit.json"
report = json.loads((root / "artifacts/ingestion" / filename).read_text(encoding="utf-8"))
print(json.dumps(report.get("test", report.get("decisions", report.get("aggregate"))), indent=2))

[
  [
    "2025-01-04",
    "EUR",
    2550,
    1
  ],
  [
    "2025-01-04",
    "USD",
    12010,
    1
  ]
]


In [2]:
import pandas as pd
result = pd.read_csv(root / "artifacts/ingestion/daily_revenue.csv")
print(result.head(8).to_string(index=False))

       day currency  amount_minor  event_count
2025-01-04      EUR          2550            1
2025-01-04      USD         12010            1


## Interpretation and limits

Local SQLite/WAL reference implementation with one payment schema and USD/EUR/GBP. PostgreSQL deployment, queue backoff, external delivery, load tests and additional provider contracts are future integration work. Atomicity covers this local database, not external side effects.